# Visual analysis of titles

In [1]:
import random

import pandas as pd
import numpy as np
from tqdm import tqdm
import json

In [2]:
df = pd.read_csv('../../Amazon_Sports_and_Outdoors/Amazon_Sports_and_Outdoors.title', sep='\t')
df.head(5)

,ent_id:token,ent_emb:float_seq
0,884509,-0.18853039269413344 0.0677280029458144 0.0227...
1,561856,-0.10877202087920926 -0.19504738371167724 -0.2...
2,239749,0.36511265353420264 0.13893028611081343 -0.135...
3,55030,-0.04588572487050822 -0.18943352195089447 0.13...
4,1277121,0.31423742583011693 -0.005561311444394548 -0.0...


### Preprocess embeddings

In [3]:
df.rename(columns={"ent_emb:float_seq": "emb", "ent_id:token": "id"}, inplace=True)
df["emb"] = df["emb"].apply(lambda x: np.fromstring(x, dtype=np.float32, sep=' '))
df.head(5)

,id,emb
0,884509,"[-0.18853039, 0.067728005, 0.02278661, -0.0040..."
1,561856,"[-0.108772025, -0.19504738, -0.2753351, -0.094..."
2,239749,"[0.36511266, 0.13893029, -0.13537216, 0.026942..."
3,55030,"[-0.045885723, -0.18943352, 0.13069113, 0.2566..."
4,1277121,"[0.31423742, -0.0055613113, -0.05842449, 0.069..."


### Fit KNN with static embeddings

In [4]:
from sklearn.neighbors import NearestNeighbors

X = np.vstack(df["emb"].values)
knn = NearestNeighbors(n_neighbors=10, metric='cosine')
knn.fit(X)

NearestNeighbors(metric='cosine', n_neighbors=10)

### Take random item and find its neighbours

In [75]:
import random

n_ids = df["id"].shape[0]

while True:
    random_idx = random.choice(range(n_ids))

    if sum(X[random_idx]) != 0:
        break

orig_id = df["id"].iloc[random_idx]

dist, nearest_ids = knn.kneighbors([X[random_idx]])

print(orig_id)

160361


### Convert into correct metadata

In [32]:
ids = list(nearest_ids[0])
distances = dist[0]

print(ids)

[138471, 551394, 885944, 1102117, 1008302, 1432288, 1137616, 1515172, 988683, 450290]


In [74]:
import csv

data = {}
orig_title = ''
with open('../../Amazon_Sports_and_Outdoors/Amazon_Sports_and_Outdoors.item') as f:
    reader = csv.reader(f, delimiter='\t')
    _ = next(reader)
    for line in reader:
        idx = int(line[0])
        title = line[1]

        if idx == orig_id:
            orig_title = title
            continue

        if idx not in ids:
            continue

        data[idx] = title

### Original title

In [78]:
print(orig_title)

Poolmaster 72724 Outdoor Jr. Table Tennis Game, Blue


### Supposedly similar titles (sorted by distance)

In [83]:
for idx in ids:
    print(f"{data[idx]}\n")

My Grandson is an eagle scout decal sticker

Valor Fitness PY-1 Portable LAT Pull Down Machine

Primos A-Frame Triple with Diamond Cut Call

Elite Fan Shop NCAA Men's Crewneck Sweatshirt A4354

Owner American 5666-109 Stinger-66 Treble Hook, Size 1, Short Shank, 4X

English Western Horse Nylon Bridle Halter Bridle Combo Pull REINS Lime 60111LGC

Glow in The Dark Basketball Net with 9.8 ft 30 LED Basketball Hoop Outdoor Basketball Rim Replacement 12 Loops Standard Size Basketball Net with Remote Control for Kids Adults Outdoor Sports School

Indianapolis Colts Football Helmet Antenna Topper

Prologo Kappa Evo Dea Saddle Matt Black 2016

Nathan HPL 020 Vest

